In [1]:
import os
import cv2
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [2]:
def knn_predict(x_train, y_train, x_test, k=5):
    distances = np.sqrt(np.sum((x_train - x_test) ** 2, axis=1)) # Khoảng cách Euclidean
    k_nearest_indices = np.argsort(distances)[:k]           # Lấy chỉ số của k điểm gần nhất
    k_nearest_labels = [y_train[i] for i in k_nearest_indices]      # đếm nhãn xuất hiện nhiều nhất
    label, count = np.unique(k_nearest_labels, return_counts=True)
    return label[np.argmax(count)]

In [3]:
DATASET_DIR = os.path.join(os.getcwd(), 'dataset')  # Load dữ liệu đã thu thập ở Bài tập 2
with open(os.path.join(DATASET_DIR, 'faces.pkl'), 'rb') as f:
    faces = pickle.load(f)
with open(os.path.join(DATASET_DIR, 'names.pkl'), 'rb') as f:
    labels = pickle.load(f)
print('Shape of Faces matrix --> ', faces.shape)
print('Labels:', labels)

Shape of Faces matrix -->  (60, 2500)
Labels: ['Khac Binh', 'Khac Binh', 'Khac Binh', 'Khac Binh', 'Khac Binh', 'Khac Binh', 'Khac Binh', 'Khac Binh', 'Khac Binh', 'Khac Binh', 'shin', 'shin', 'shin', 'shin', 'shin', 'shin', 'shin', 'shin', 'shin', 'shin', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'shinbkb', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh', 'khacbinh']


In [4]:
facecascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)
camera = cv2.VideoCapture(0)
while True:
    ret, frame = camera.read()
    if ret == True:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        face_coordinates = facecascade.detectMultiScale(gray, 1.3, 5)
        for (a, b, w, h) in face_coordinates:
            # Cắt vùng mặt → resize 50x50 → flatten
            fc = gray[b:b + h, a:a + w]
            r  = cv2.resize(fc, (50, 50)).flatten().reshape(1, -1)
            # Dự đoán bằng KNN 
            text = knn_predict(faces, labels, r[0], k=5)
            cv2.putText(frame, text, (a, b - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
            cv2.rectangle(frame, (a, b), (a + w, b + w), (0, 0, 255), 2)
        cv2.imshow('livetime face recognition', frame)
        if cv2.waitKey(1) == 27:   # ESC để thoát
            break
    else:
        print('error')
        break
cv2.destroyAllWindows()
camera.release()